In [1]:
import joblib
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import accuracy_score, f1_score, classification_report
import pandas as pd

In [2]:
# Load data
X_train = joblib.load("X_train.pkl")
X_test = joblib.load("X_test.pkl")
y_train = joblib.load("y_train.pkl")
y_test = joblib.load("y_test.pkl")

print("X_test shape:", X_test.shape)
print("y_test shape:", y_test.shape)
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

X_test shape: (412, 4)
y_test shape: (412,)
X_train shape: (960, 4)
y_train shape: (960,)


In [5]:
print("Model được chọn để tuning: RandomForest")
print("Lý do: Đây là model có F1-score cao nhất và ổn định nhất từ bước so sánh trước.")

# ====================== 2. Tuning hyperparameters ======================
param_grid = {
    'n_estimators': [100, 200, 300],
    'max_depth': [None, 15, 20, 25],
    'min_samples_split': [2, 5, 10],
    'min_samples_leaf': [1, 2, 4],
    'max_features': ['sqrt', 'log2']
}

print("\nĐang thực hiện GridSearchCV cho RandomForest...")

grid_search = GridSearchCV(
    estimator=RandomForestClassifier(random_state=42),
    param_grid=param_grid,
    cv=5,
    scoring='f1',
    n_jobs=-1,
    verbose=1
)

grid_search.fit(X_train, y_train)

print("\nBest parameters:", grid_search.best_params_)
print("Best CV F1-score:", round(grid_search.best_score_, 4))

# Lấy model tốt nhất
best_rf = grid_search.best_estimator_

# 3. Train lại model sau tuning
y_pred_tuned = best_rf.predict(X_test)

# Lưu model bằng joblib (khuyến nghị cho scikit-learn)
joblib.dump(best_rf, 'RandomForest_tuned.joblib')
print("\nĐã lưu model sau tuning: RandomForest_tuned.joblib")

# 4. Đánh giá lại model 
print("\n=== Đánh giá RandomForest sau Tuning ===")
print(classification_report(y_test, y_pred_tuned))

acc = accuracy_score(y_test, y_pred_tuned)
f1 = f1_score(y_test, y_pred_tuned)

print(f"Accuracy  : {acc:.4f}")
print(f"F1-score  : {f1:.4f}")

# --- Kết quả TRƯỚC tuning (dùng model mặc định) ---
rf_default = RandomForestClassifier(random_state=42)
rf_default.fit(X_train, y_train)
y_pred_before = rf_default.predict(X_test)

acc_before = accuracy_score(y_test, y_pred_before)
f1_before = f1_score(y_test, y_pred_before)

# --- Kết quả SAU tuning ---
acc_after = accuracy_score(y_test, y_pred_tuned)
f1_after = f1_score(y_test, y_pred_tuned)

print("=== So sánh trước và sau tuning ===")
print(f"{'Metric':<15} {'Trước Tuning':<15} {'Sau Tuning':<15} {'Cải thiện':<10}")
print("-" * 55)
print(f"{'Accuracy':<15} {acc_before:.4f}{'':<7} {acc_after:.4f}{'':<7} {acc_after - acc_before:+.4f}")
print(f"{'F1-score':<15} {f1_before:.4f}{'':<7} {f1_after:.4f}{'':<7} {f1_after - f1_before:+.4f}")

print("\n=== Classification Report sau Tuning ===")
print(classification_report(y_test, y_pred_tuned))

Model được chọn để tuning: RandomForest
Lý do: Đây là model có F1-score cao nhất và ổn định nhất từ bước so sánh trước.

Đang thực hiện GridSearchCV cho RandomForest...
Fitting 5 folds for each of 216 candidates, totalling 1080 fits

Best parameters: {'max_depth': None, 'max_features': 'sqrt', 'min_samples_leaf': 1, 'min_samples_split': 2, 'n_estimators': 100}
Best CV F1-score: 0.9908

Đã lưu model sau tuning: RandomForest_tuned.joblib

=== Đánh giá RandomForest sau Tuning ===
              precision    recall  f1-score   support

           0       1.00      1.00      1.00       229
           1       1.00      0.99      1.00       183

    accuracy                           1.00       412
   macro avg       1.00      1.00      1.00       412
weighted avg       1.00      1.00      1.00       412

Accuracy  : 0.9976
F1-score  : 0.9973
=== So sánh trước và sau tuning ===
Metric          Trước Tuning    Sau Tuning      Cải thiện 
-------------------------------------------------------
Ac